In [1]:
import os


In [2]:
%pwd

'c:\\Users\\H_KhaN\\OneDrive\\Desktop\\Mlops_project_wine_quality\\research'

In [4]:
os.chdir("../")

In [5]:
%pwd

'c:\\Users\\H_KhaN\\OneDrive\\Desktop\\Mlops_project_wine_quality'

In [19]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelTrainerConfig:
    """
    Typed container for everything the ModelTrainer stage needs.
    Built by ConfigurationManager.get_model_trainer() and passed into ModelTrainer.
    """

    root_dir: Path          # Folder where the trained model will be saved
    train_data_path: Path   # Path to the training CSV
    test_data_path: Path    # Path to the test CSV
    model_name: str         # Filename of the saved model (e.g., model.joblib)
    alpha: float            # ElasticNet overall regularization strength
    l1_ratio: float         # ElasticNet mix of L1 (Lasso) vs L2 (Ridge) penalty
    target_column: str      # Name of the column the model should predict

In [20]:
from src.Mlops_project_quality_predictor.constants import *
from src.Mlops_project_quality_predictor.utils.common import read_yaml,create_directories



In [21]:
class ConfigurationManager:
    """
    Central place to load project configuration and build typed config
    objects (e.g., DataTransformationConfig) for each pipeline stage.
    """

    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,   # Path to config.yaml (paths, directories, artifacts)
        params_filepath=PARAMS_FILE_PATH,   # Path to params.yaml (model/hyperparameters)
        schema_filepath=SCHEMA_FILE_PATH):  # Path to schema.yaml (expected columns & dtypes)

        # Read each YAML file and store it as an attribute.
        # read_yaml returns a ConfigBox so keys can be accessed
        # with dot notation (e.g., self.config.data_transformation)
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        # Make sure the top-level artifacts folder exists before any stage runs
        create_directories([self.config.artifacts_root])


    def get_model_trainer(self) -> ModelTrainerConfig:
            """
            Build and return the ModelTrainerConfig used by the ModelTrainer stage.
            Combines paths from config.yaml, hyperparameters from params.yaml,
            and the target column name from schema.yaml.
            """
            # Section of config.yaml with paths for the model trainer stage
            config = self.config.model_trainer

            # ElasticNet hyperparameters (alpha, l1_ratio) from params.yaml
            params = self.params.ElasticNet

            # Target column definition from schema.yaml
            schema = self.schema.TARGET

            # Create the model trainer's output directory if it doesn't exist
            create_directories([config.root_dir])

            # Bundle everything into a typed config object
            model_trainer_config = ModelTrainerConfig(
                root_dir=config.root_dir,                  # Where the trained model will be saved
                train_data_path=config.train_data_path,    # Path to the training CSV
                test_data_path=config.test_data_path,      # Path to the test CSV
                model_name=config.model_name,              # Filename for the saved model (e.g., model.joblib)
                alpha=params.alpha,                        # Regularization strength
                l1_ratio=params.l1_ratio,                  # L1 vs L2 penalty mix
                target_column=schema.name,                 # Name of the column to predict
            )

            return model_trainer_config

In [22]:
import pandas as pd
import os
from src.Mlops_project_quality_predictor.utils import logger
from sklearn.linear_model import ElasticNet
import joblib

In [23]:

class ModelTrainer:
    """Trains an ElasticNet regression model and saves it to disk."""

    def __init__(self, config: ModelTrainerConfig):
        # Store the config (data paths, hyperparameters, output location)
        self.config = config

    def train(self):
        # Load the training and test datasets from CSV files
        train_data = pd.read_csv(self.config.train_data_path)
        test_data = pd.read_csv(self.config.test_data_path)

        # Split training data into features (X) and target (y)
        train_x = train_data.drop([self.config.target_column], axis=1)
        train_y = train_data[self.config.target_column]

        # Split test data the same way
        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[self.config.target_column]

        # Create the ElasticNet model using hyperparameters from the config
        # alpha    -> overall regularization strength
        # l1_ratio -> mix of L1 (Lasso) vs L2 (Ridge) penalty
        # random_state -> makes results reproducible
        lr = ElasticNet(
            alpha=self.config.alpha,
            l1_ratio=self.config.l1_ratio,
            random_state=42,
        )

        # Fit the model on the training data
        lr.fit(train_x, train_y)

        # Save the trained model to disk so it can be reused later
        joblib.dump(lr, os.path.join(self.config.root_dir, self.config.model_name))

In [24]:
try:
    # Load config.yaml, params.yaml and schema.yaml and set up the artifacts folder
    config = ConfigurationManager()

    # Build the typed config object for the model trainer stage
    model_trainer_config = config.get_model_trainer()

    # Create the trainer, giving it the config it needs
    model_trainer = ModelTrainer(config=model_trainer_config)

    # Load the data, train the ElasticNet model, and save it to disk
    model_trainer.train()

except Exception as e:
    # Log the failure with the full traceback, then re-raise so the pipeline stops
    logger.exception(f"Model trainer stage failed: {e}")
    raise e

[2026-10-15 21:50:31,332:INFO:common:YAML file loaded successfully: C:\Users\H_KhaN\OneDrive\Desktop\Mlops_project_wine_quality\config\config.yaml]
[2026-10-15 21:50:31,335:INFO:common:YAML file loaded successfully: C:\Users\H_KhaN\OneDrive\Desktop\Mlops_project_wine_quality\params.yaml]
[2026-10-15 21:50:31,341:INFO:common:YAML file loaded successfully: C:\Users\H_KhaN\OneDrive\Desktop\Mlops_project_wine_quality\schema.yaml]
[2026-10-15 21:50:31,342:INFO:common:created directory at: artifacts]
[2026-10-15 21:50:31,345:INFO:common:created directory at: artifacts/model_trainer]


In [18]:
print(SCHEMA_FILE_PATH)
cfg = ConfigurationManager()
print(cfg.schema.keys())

schema.yaml
[2026-10-15 21:49:08,772:INFO:common:YAML file loaded successfully: C:\Users\H_KhaN\OneDrive\Desktop\Mlops_project_wine_quality\config\config.yaml]
[2026-10-15 21:49:08,776:INFO:common:YAML file loaded successfully: C:\Users\H_KhaN\OneDrive\Desktop\Mlops_project_wine_quality\params.yaml]
[2026-10-15 21:49:08,780:INFO:common:YAML file loaded successfully: C:\Users\H_KhaN\OneDrive\Desktop\Mlops_project_wine_quality\schema.yaml]
[2026-10-15 21:49:08,783:INFO:common:created directory at: artifacts]
dict_keys(['COLUMNS', 'TARGET'])
